# cache

> Thin helpers for cache paths and generic model loading

In [ ]:
#| default_exp cache

## Design

This module centralizes path construction and generic JSON→Pydantic loading for cached artifacts.

It deliberately does **not** own refresh or invalidation policy. Those remain in stage modules like `toc.py` and `summarize.py`.

It also does **not** import downstream models such as `TocFile` or `AssembledSummaries`. Callers pass their own model class into `read_model(...)` to avoid circular imports.

In [ ]:
#| export
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import TypeVar
from pydantic import BaseModel
from yttoc.core import Meta

TModel = TypeVar('TModel', bound=BaseModel)

_DEFAULT_ROOT = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'yttoc'


In [ ]:
#| export
def resolve_root(root: str | Path | None = None # Explicit root override
                ) -> Path: # Resolved cache root
    "Return the effective cache root as a Path."
    return Path(root) if root else _DEFAULT_ROOT

def video_dir(video_id: str, # Exact video_id
              root: str | Path | None = None # Cache root override
             ) -> Path: # Video cache directory
    "Return the cache directory for one video."
    return resolve_root(root) / video_id

def meta_path(video_id: str, # Exact video_id
              root: str | Path | None = None # Cache root override
             ) -> Path: # Path to meta.json
    "Return the path to meta.json for one video."
    return video_dir(video_id, root) / 'meta.json'

def toc_path(video_id: str, # Exact video_id
             root: str | Path | None = None # Cache root override
            ) -> Path: # Path to toc.json
    "Return the path to toc.json for one video."
    return video_dir(video_id, root) / 'toc.json'

def summaries_path(video_id: str, # Exact video_id
                   root: str | Path | None = None # Cache root override
                  ) -> Path: # Path to summaries.json
    "Return the path to summaries.json for one video."
    return video_dir(video_id, root) / 'summaries.json'

def glob_srt(out_dir: str | Path, # Directory to search
             pattern: str = 'captions.*.srt' # Glob pattern
            ) -> list[Path]: # Sorted matching paths
    "Find SRT files matching a glob pattern."
    return sorted(Path(out_dir).glob(pattern))

def first_srt_path(video_id: str, # Exact video_id
                   root: str | Path | None = None # Cache root override
                  ) -> Path: # First matching captions.*.srt path
    "Return the first cached SRT path for one video or raise if none exist."
    matches = glob_srt(video_dir(video_id, root))
    if not matches:
        raise FileNotFoundError(f'No captions found for {video_id}')
    return matches[0]

def read_model(path: str | Path, # JSON file path
               model_cls: type[TModel] # Pydantic model class
              ) -> TModel: # Parsed model instance
    "Read a JSON file and validate it with the caller-provided Pydantic model class."
    path = Path(path)
    return model_cls.model_validate_json(path.read_text(encoding='utf-8'))

def load_meta(video_id: str, # Exact video_id
              root: str | Path | None = None # Cache root override
             ) -> Meta: # Parsed Meta instance
    "Load meta.json for one cached video."
    return read_model(meta_path(video_id, root), Meta)

def touch_meta(video_id: str, # Exact video_id
               root: str | Path | None = None # Cache root override
              ) -> None:
    "Bump last_used_at on meta.json for one cached video."
    path = meta_path(video_id, root)
    meta = read_model(path, Meta)
    meta.last_used_at = datetime.now(timezone.utc)
    path.write_text(meta.model_dump_json(indent=2), encoding='utf-8')


## Tests

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    assert resolve_root(root) == root
    assert video_dir('abc', root) == root / 'abc'
    assert meta_path('abc', root) == root / 'abc' / 'meta.json'
    assert toc_path('abc', root) == root / 'abc' / 'toc.json'
    assert summaries_path('abc', root) == root / 'abc' / 'summaries.json'
print('ok')


In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    vdir = root / 'abc'
    vdir.mkdir()
    second = vdir / 'captions.ja.srt'
    first = vdir / 'captions.en.srt'
    second.write_text('2', encoding='utf-8')
    first.write_text('1', encoding='utf-8')
    assert first_srt_path('abc', root) == first

with TemporaryDirectory() as d:
    try:
        first_srt_path('missing', d)
    except FileNotFoundError:
        pass
    else:
        assert False, 'expected FileNotFoundError when no captions exist'
print('ok')


In [ ]:
from datetime import datetime, timezone
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    vdir = root / 'vid'
    vdir.mkdir()
    meta = Meta(
        id='vid', title='T', channel='C', duration=60,
        upload_date='20260101', webpage_url='u',
        description='', captions={'en': 'auto'},
        last_used_at=datetime(2026, 1, 1, tzinfo=timezone.utc))
    (vdir / 'meta.json').write_text(meta.model_dump_json(), encoding='utf-8')
    loaded = load_meta('vid', root)
    assert loaded == meta
    loaded2 = read_model(vdir / 'meta.json', Meta)
    assert loaded2 == meta
print('ok')


In [ ]:
import json, time
from datetime import datetime
from tempfile import TemporaryDirectory

# glob_srt: sorting + pattern override
with TemporaryDirectory() as d:
    base = Path(d)
    assert glob_srt(base) == []
    (base / 'captions.ja.srt').write_text('2', encoding='utf-8')
    (base / 'captions.en.srt').write_text('1', encoding='utf-8')
    assert glob_srt(base) == [base / 'captions.en.srt', base / 'captions.ja.srt']
    (base / 'captions_en.en-US.srt').write_text('x', encoding='utf-8')
    assert glob_srt(base, 'captions_en*.srt') == [base / 'captions_en.en-US.srt']

# touch_meta: bumps last_used_at monotonically
with TemporaryDirectory() as d:
    root = Path(d)
    vdir = root / 'TM1'
    vdir.mkdir()
    (vdir / 'meta.json').write_text(json.dumps({
        'id': 'TM1', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'u',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }), encoding='utf-8')
    touch_meta('TM1', root)
    first = json.loads((vdir / 'meta.json').read_text())['last_used_at']
    assert first != '2000-01-01T00:00:00+00:00'
    first_dt = datetime.fromisoformat(first)
    assert first_dt.tzinfo is not None
    time.sleep(0.001)
    touch_meta('TM1', root)
    second_dt = datetime.fromisoformat(json.loads((vdir / 'meta.json').read_text())['last_used_at'])
    assert second_dt >= first_dt
print('ok')
